# Saving & loading models

Training is expensive; inference should be cheap. Once you've trained, tuned,
and [explained](../07-explainability/model-interpretability.ipynb) a model, you
want to **persist** it — write it to disk and reload it later to serve
predictions without retraining.

In Rust the idiomatic path is [`serde`](https://serde.rs) for the
serialize/deserialize traits plus a binary format like
[`bincode`](https://docs.rs/bincode). Any model type that derives
`Serialize`/`Deserialize` can be saved.

In [ ]:
:dep serde = { version = "1", features = ["derive"] }
:dep bincode = { version = "1.3" }

// A trained model = its learned parameters. This mirrors the linear-regression
// chapter (weights + bias). Deriving serde traits makes it persistable.
#[derive(serde::Serialize, serde::Deserialize, Clone, Debug)]
struct LinearModel {
    weights: Vec<f64>,
    bias: f64,
}
impl LinearModel {
    fn predict(&self, x: &[f64]) -> f64 {
        self.bias + self.weights.iter().zip(x).map(|(w, xi)| w * xi).sum::<f64>()
    }
}

let model = LinearModel { weights: vec![2.0, -1.0], bias: 0.5 };
let sample = [3.0, 4.0];
println!("prediction before saving = {:.3}", model.predict(&sample));

## Round-trip through disk

Serialize to bytes, write to a file, read it back, deserialize — then confirm
the reloaded model makes the **identical** prediction:

In [ ]:
{
    let path = "/tmp/linear_model.bin";
    // Save
    let bytes = bincode::serialize(&model).unwrap();
    std::fs::write(path, &bytes).unwrap();
    println!("wrote {} bytes to {}", bytes.len(), path);
    // Load
    let loaded_bytes = std::fs::read(path).unwrap();
    let loaded: LinearModel = bincode::deserialize(&loaded_bytes).unwrap();
    println!("prediction after loading = {:.3}", loaded.predict(&sample));
    println!("match: {}", (model.predict(&sample) - loaded.predict(&sample)).abs() < 1e-12);
}

## Versioning models on disk

Don't silently overwrite a working model. Save **metadata** alongside it — a
version tag, the training date, and something about the data (row count or a
hash) — so you can tell models apart and roll back:

In [ ]:
#[derive(serde::Serialize, serde::Deserialize, Debug)]
struct ModelArtifact {
    version: u32,
    trained_on: String,
    n_train_rows: usize,
    model: LinearModel,
}

{
    let artifact = ModelArtifact {
        version: 1,
        trained_on: "2026-01-15".to_string(),
        n_train_rows: 5000,
        model: model.clone(),
    };
    let bytes = bincode::serialize(&artifact).unwrap();
    std::fs::write("/tmp/model_v1.bin", &bytes).unwrap();
    let reloaded: ModelArtifact = bincode::deserialize(&std::fs::read("/tmp/model_v1.bin").unwrap()).unwrap();
    println!("loaded artifact v{} trained {} on {} rows",
             reloaded.version, reloaded.trained_on, reloaded.n_train_rows);
}

## Interoperability: ONNX

If you need to consume a model **trained in another stack** (PyTorch, scikit-
learn via `skl2onnx`, etc.), the [`tract`](https://docs.rs/tract-onnx) crate
loads and runs `.onnx` files directly in Rust — useful for shipping a
Python-trained model inside a Rust service.

```{note}
The other direction is weaker: `smartcore`/`linfa` models **don't export to ONNX
natively**, so within this ecosystem, staying end-to-end in Rust with
`serde`+`bincode` (as above) is the reliable persistence story. ONNX is
realistically for *importing* models from elsewhere, or for gradient-boosted
models via crates that support ONNX export.
```

Next: [serving a model](serving-a-model.ipynb) — loading a persisted model behind
an HTTP endpoint.